# Week 100, Day 4: Mock loop

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w100d4.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import math
total = 0.0
for v in [0.1] * 10:
    total += v
print(total, total == 1.0, math.fsum([0.1] * 10) == 1.0)

*Interactive figure on the website: Two and a half hours for the whole loop..*

### Exercise: Coding 1: A grid that follows the rules

A 9 by 9 puzzle grid is a list of 9 strings of digits and `"."` for blanks. Write `valid_grid(rows)`: whether no digit repeats in any row, any column, or any of the nine 3 by 3 boxes (blanks don't count). One pass, with sets.

In [ ]:
def valid_grid(rows):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = [\"53..7....\", \"6..195...\", \".98....6.\", \"8...6...3\", \"4..8.3..1\", \"7...2...6\", \".6....28.\", \"...419..5\", \"....8..79\"]\n\ndef test_grid():\n    \"rows, columns, boxes\"\n    assert valid_grid(GOOD)\n    assert not valid_grid([\"83..7....\"] + GOOD[1:]), \"two 8s in a column\"\n    assert not valid_grid([\"55..7....\"] + GOOD[1:]), \"two 5s in a row\"\n    box = GOOD[:]\n    box[1] = \"6.3195...\"\n    assert not valid_grid(box), \"two 3s in a box\"\n    assert valid_grid([\".\" * 9] * 9)")

### Exercise: Coding 2: Events in the last minute

Write a class `RecentCounter(window=60)` whose `hit(t)` records an event at time `t` (seconds, never decreasing) and returns how many events happened in the window `[t - window + 1, t]`. Each call must be amortised O(1).

In [ ]:
from collections import deque

class RecentCounter:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import time\n\ndef test_counter():\n    \"a sliding window of times\"\n    c = RecentCounter(window=60)\n    assert [c.hit(t) for t in (1, 30, 60, 61, 120, 121)] == [1, 2, 3, 3, 2, 2]\n\ndef test_fast():\n    \"a million hits\"\n    c = RecentCounter(window=1000)\n    start = time.perf_counter()\n    for t in range(1_000_000):\n        last = c.hit(t)\n    assert last == 1000 and time.perf_counter() - start < 3")

### Exercise: Design: finding junctions nearby

Waymark finds junctions near a walker by splitting the map into square cells. Write `nearby_plan(junctions, area_km2, radius_km, cell_km)`: return a dict with `cells_searched` (the square of cells covering the circle: `(2 × ceil(radius / cell) + 1) ** 2`), `per_cell` (junctions per cell on average: junctions × cell area / area, 2 decimal places) and `candidates` (cells searched × per cell, rounded to a whole number), and `advice`: `"cells too big"` if `per_cell` is over 100, `"cells too small"` if `cells_searched` is over 200, else `"ok"`.

In [ ]:
import math

def nearby_plan(junctions, area_km2, radius_km, cell_km):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_plan():\n    \"a grid index\"\n    assert nearby_plan(1_000_000, 250_000, 2, 1) == {\"cells_searched\": 25, \"per_cell\": 4.0, \"candidates\": 100, \"advice\": \"ok\"}\n    assert nearby_plan(1_000_000, 250_000, 2, 10)[\"advice\"] == \"cells too big\"\n    assert nearby_plan(1_000_000, 250_000, 10, 0.5)[\"advice\"] == \"cells too small\"")

### Bug hunt: A total that drifted

Adding ten 0.1 km segments one at a time with `+=` gives `0.9999999999999999`, so the check that a route is exactly 1 km fails and this prints `False`. Use `math.fsum` so it prints `True`.

In [ ]:
import math

segments = [0.1] * 10
total = 0.0
for s in segments:
    total += s
print(total == 1.0)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"correctly rounded sums\"\n    expect_output(\"True\")")

### Bug hunt: The last junction never checked

`range(len(heights) - 1)` stops one short, so the highest junction (the last) is missed and this prints `120`. Check every junction so it prints `310`.

In [ ]:
heights = [40, 55, 120, 310]
highest = heights[0]
for i in range(len(heights) - 1):
    highest = max(highest, heights[i])
print(highest)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"every index\"\n    expect_output(\"310\")")

### Exercise: Debugging: an order that wasn't kept

`unique_stops(stops)` should remove repeated stops keeping the **first** occurrence order, but `list(set(...))` loses the order. Fix it.

In [ ]:
def unique_stops(stops):
    return list(set(stops))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_unique():\n    \"first-seen order\"\n    assert unique_stops([\"Mill\", \"Weir\", \"Mill\", \"Ridge\", \"Weir\"]) == [\"Mill\", \"Weir\", \"Ridge\"]\n    assert unique_stops([]) == [] and unique_stops([\"z\", \"a\", \"z\"]) == [\"z\", \"a\"]")

### Exercise: Deep dive: a running average you send values to

Write a generator function `averager()`: after it's started with `next()`, each `send(value)` returns the average of all values sent so far, as a float. Then write `averages(values)`, using `averager` to return the list of running averages.

In [ ]:
def averager():
    ___

def averages(values):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import inspect\n\ndef test_averager():\n    \"send() resumes the generator\"\n    gen = averager()\n    assert inspect.isgenerator(gen)\n    next(gen)\n    assert gen.send(10) == 10.0 and gen.send(20) == 15.0 and gen.send(0) == 10.0\n    assert averages([6, 7.5, 12]) == [6.0, 6.75, 8.5] and averages([]) == []")

> **Behavioural: a time you had to learn something fast.** Two minutes, aloud: "Tell me about a time you had to learn something quickly to get a job done." Building CPython in Week 94 is a good story.

### Puzzle: Three kinds of key

Arrange the loop of `valid_grid`. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the box (a tuple) that cell row 4, column 7 belongs to
q = ___   # RecentCounter(10) hits at 1, 5, 11, 12: the last returned count
r = ___   # list(dict.fromkeys("banana"))
s = ___   # 0.1 + 0.2 == 0.3
t = ___   # what a generator's first next() returns in averager (the first yielded value)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == (1, 2) and q == 3 and r == [\"b\", \"a\", \"n\"], \"p, q and r\"\n    assert s is False and t is None, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

Adding many floats one at a time with `+=` accumulates tiny rounding errors; `math.fsum` tracks them and gives the correctly rounded total. (Since Python 3.12 the built-in `sum` also compensates for them, but a hand-written loop doesn't.) One of today's bugs lives here.

## The loop

### Coding round

### Design round

### Debugging round

### Python deep-dive round

### Behavioural round

## Quiz

## Scoring yourself

Score each round and add them to your log. One more loop tomorrow.

## Before you go

Tomorrow is the last mock loop of the course.